# PyCAM-SIMA dynamic persistent model pool

This Notebook starts one dynamically sized MPI world and splits it into reusable model slots. Forked models copy rank-local StatePool data directly inside that MPI world: no child `qsub`, no child `mpiexec`, and no checkpoint file is required.

## 1. Execution model

```text
Jupyter → Dask Actor → one mpiexec
                         │
                         ├── slot 0: base
                         ├── slot 1: child
                         └── slot N: child / idle

base.fork(...) → matching MPI ranks copy Python-owned StatePool arrays
```

`ranks_per_model` defaults to `ModelConfig.mpi_size`. The number of slots is calculated from the available PBS CPU and memory resources; neither value is hard-coded.

## 2. Configure Dask and inspect the resource plan

In [ ]:
from datetime import datetime
from pathlib import Path
import os
import shutil

import numpy as np
import pycam_sima
from dask.distributed import Client
from pycam_sima import DaskExperimentClient

repo = Path('/glade/work/ruitong/pycam-sima')
scratch = Path(os.environ.get('SCRATCH', '/glade/derecho/scratch/ruitong'))
config_path = repo / 'configs/fkessler_model.yaml'
reference_atm_in = repo / 'reference/cases/FKESSLER_ne3pg3_gnu_24x50/CaseDocs/atm_in'
stamp = datetime.now().strftime('%Y%m%d-%H%M%S')
experiment_root = scratch / 'pycam-sima/persistent_pool_trials' / stamp
initial_run_dir = experiment_root / 'initial-run'
initial_run_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(reference_atm_in, initial_run_dir / 'atm_in')

# Run this Notebook inside a PBS allocation for a local, single-allocation pool.
execution_mode = 'allocation' if os.environ.get('PBS_JOBID') else 'pbs'
client = Client(processes=False, n_workers=1, threads_per_worker=1, dashboard_address=None)
experiments = DaskExperimentClient(
    client,
    config=config_path,
    initial_run_dir=initial_run_dir,
    run_root=experiment_root / 'models',
    python_executable=repo / '.venv/bin/python',
    execution_mode=execution_mode,
)

# None inherits mpi_size from the model config. Change it to an integer or 'auto'.
resource_plan = experiments.plan_pool(
    max_concurrent_models=4,
    ranks_per_model=None,
    memory_per_model='auto',
)
resource_plan.describe()

## 3. One pool, one base, and in-memory branches

This is the only pool creation in the Notebook. Closing a child returns its slot; closing the outer `with` stops the one MPI world.

In [ ]:
try:
    with experiments.pool(
        name='cam-pool',
        resource_plan=resource_plan,
    ) as pool:
        with pool.model('base') as base:
            temperature_before = base.fields.air_temperature.stats(rank=0)
            base.advance(steps=2)

            # Dynamically add a Python-owned StatePool variable before forking.
            base.fields.create(
                'experiment_tracer',
                dims=('column', 'level'),
                units='kg kg-1',
                initial=0.0,
            )

            # Each child receives a private, bitwise copy of the base arrays.
            with base.fork('control', 'no_kessler', 'warm') as branches:
                branches.no_kessler.physics.kessler.enabled = False
                branches.warm.fields.air_temperature += 1.0

                control_initial = branches.control.fields.air_temperature.get(rank=0)
                warm_initial = branches.warm.fields.air_temperature.get(rank=0)
                assert np.array_equal(warm_initial, np.add(control_initial, 1.0))

                # One pool command advances all occupied child slots concurrently.
                branches.advance(steps=1)
                branch_statuses = branches.statuses

            pool_status = pool.status
            final_base_status = base.status

        result = {
            'resource_plan': resource_plan.describe(),
            'pool_mpi_launch_count': pool_status['mpi_launch_count'],
            'base_step': final_base_status.step,
            'temperature_before': temperature_before,
            'branch_steps': {name: status.step for name, status in branch_statuses.items()},
            'slots_after_children_close': pool.slots,
        }
finally:
    client.close()

result